In [48]:
!pip install groq

In [49]:
import pandas as pd
import numpy as np
import shap
import xgboost as xgb
import joblib
import os
import logging
from typing import Dict, Any, List
from groq import Groq

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("DeliriumOptimizationAgent")

In [50]:
import os

# Paste your Groq API key here
GROQ_API_KEY = "your_api_key_here"

# Set environment variable
os.environ['GROQ_KEY'] = GROQ_API_KEY

print("Groq API Key set successfully.")

Groq API Key set successfully.


In [51]:
def load_resources(model_path: str = 'deliriumwatch_ensemble.pkl', data_path: str = 'deliriumwatch_clean_tv.csv'):
    """Loads the pre-trained model and a sample dataset."""
    model = None
    df = None
    try:
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            logger.info(f"Model loaded from {model_path}.")
        else:
            logger.warning(f"Model not found at {model_path}! Mocking model for demo.")
            # Mocking standard XGB struct for demonstration purposes 
            class MockModel:
                def predict_proba(self, X): return np.array([[0.3, 0.7]] * len(X))
            model = MockModel()
            
        if os.path.exists(data_path):
            df = pd.read_csv(data_path)
            logger.info(f"Dataset loaded with {df.shape[0]} rows.")
    except Exception as e:
        logger.error(f"Error loading resources: {str(e)}")
        
    return model, df

model, sample_df = load_resources()

2026-03-30 23:33:09 [INFO] Model loaded from deliriumwatch_ensemble.pkl.


In [52]:
class DeliriumOptimizationAgent:
    """
    Intelligent Delirium Risk Optimization Agent
    - Computes SHAP importance
    - Filters modifiable/intervenable criteria
    - Applies CPU-efficient hybrid constraint optimization
    """
    INTERVENABLE_FEATURES = [
        'hr_mean', 'hr_std',
        'rr_mean', 'rr_std',
        'spo2_mean', 'spo2_min',
        'nibp_mean',
        'propofol_total',
        'midazolam_total',
        'lorazepam_total',
        'fentanyl_total',
        'unique_sedation_drugs',
        'nid_score'
    ]

    INDIRECT_FEATURES = [
        'avg_sofa'
    ]

    NON_INTERVENABLE_FEATURES = [
        'age',
        'gender',
        'abp_measured'
    ]

    def __init__(self, model, debug: bool = False):
        self.model = model
        self.debug = debug
        try:
            # Use tree explainer from XGBoost estimator if inside an ensemble mapping
            if hasattr(self.model, 'named_estimators_'):
                self.explainer = shap.TreeExplainer(self.model.named_estimators_['xgb'])
            else:
                self.explainer = shap.TreeExplainer(self.model)
        except Exception as e:
            logger.warning(f"Could not initialize standard TreeExplainer ({e}). Yielding to generic SHAP stub.")
            self.explainer = None
            
    def _log(self, msg: str):
        if self.debug:
            logger.info(msg)

    def validate_features(self, patient_df: pd.DataFrame):
        """Basic feature validation layer."""
        if len(patient_df) != 1:
            raise ValueError("Agent operates on single-patient inputs (1 row).")
            
    def get_shap_values(self, patient_df: pd.DataFrame):
        """Computes generic SHAP attribution."""
        if not self.explainer:
            # Mock stub logic if explainer fails
            return {col: np.random.uniform(-0.1, 0.4) for col in patient_df.columns}
            
        shap_vals = self.explainer.shap_values(patient_df)
        if isinstance(shap_vals, list):
            instance_shap = shap_vals[1][0]  
        else:
            instance_shap = shap_vals[0]     
            if len(instance_shap.shape) > 1 and instance_shap.shape[1] == 2:
                instance_shap = instance_shap[:, 1]
        
        return {feat: val for feat, val in zip(patient_df.columns, instance_shap)}

    def is_modifiable(self, feature: str) -> bool:
        """Determines if feature is part of the clinical intervention list."""
        return feature in self.INTERVENABLE_FEATURES

    def generate_safe_candidates(self, feature: str, val: float) -> List[float]:
        """
        Constraint-aware generation of hypothetical safe improvements. 
        Rules:
        - No neg values
        - No jumps > 50% unless strictly allowed
        - HR: 40-180, RR: 8-40
        - SpO2: NEVER DROP, max=100
        - Sedatives: Max drop 20-30%, NO INSTANT ZERO drops
        """
        if pd.isna(val) or isinstance(val, str) or type(val) in [bool, np.bool_]:
            return []
        if val < 0:
            return []

        candidates = set()
        
        # VITAL SIGNS
        if 'hr_' in feature:
            # Attempt mild downward adjustments if HR is elevated (local search multipliers)
            if val > 80:
                for m in [0.95, 0.90, 0.85]:
                    c = val * m
                    if 40 <= c <= 180: candidates.add(c)
        elif 'rr_' in feature:
            # Mild down if elevated
            if val > 16:
                for m in [0.95, 0.90, 0.85]:
                    c = val * m
                    if 8 <= c <= 40: candidates.add(c)
        elif 'spo2' in feature:
            # NEVER decrease, attempt slight oxygenation boosts if under 100%
            if val < 100:
                for m in [1.02, 1.05]:
                    c = val * m
                    if val < c <= 100:
                        candidates.add(c)
        elif 'nibp' in feature:
            # Blood pressure: avoid unrealistic drops
            for m in [0.95, 0.90]:
                c = val * m
                if c >= 65:  # Absolute clinical safe floor for MAP estimation
                    candidates.add(c)
                    
        # SEDATIVES (CRITICAL)
        elif any(s in feature for s in ['propofol', 'midazolam', 'lorazepam', 'fentanyl']):
            if val > 0:
                # Cannot drop to 0 instantly, max reduction per step is 30%
                for m in [0.90, 0.80, 0.70]:  # 10%, 20%, 30% drops
                    candidates.add(val * m)
        elif feature == 'unique_sedation_drugs':
            if val >= 2:
                candidates.add(val - 1)  # Gradual polypharmacy reduction
        elif feature == 'nid_score':
            if val > 0:
                candidates.add(val * 0.8) # 20% reduction in sleep disruption factor

        # Global Constraints Filter
        safe_cands = []
        for c in candidates:
            if c < 0: continue
            # Enforce max 50% relative jump ceiling globally (unless unique drug count drop)
            if val > 0 and 'unique' not in feature:
                pct_jump = abs(c - val) / val
                if pct_jump > 0.50:
                    continue
            safe_cands.append(c)
            
        return sorted(safe_cands)

    def predict_risk(self, patient_df: pd.DataFrame) -> float:
        return float(self.model.predict_proba(patient_df)[0, 1])

    def process_patient(self, input_features: pd.DataFrame) -> Dict[str, Any]:
        self.validate_features(input_features)
        X = input_features.copy()

        # 1. Baseline Risk
        original_risk = self.predict_risk(X)
        self._log(f"Original Risk Computed: {original_risk:.4f}")

        # 2. Get Top Positive Risk Drivers via SHAP
        shap_dict = self.get_shap_values(X)
        pos_features = {k: v for k, v in shap_dict.items() if v > 0}
        top_drivers = [f for f, v in sorted(pos_features.items(), key=lambda i: i[1], reverse=True)][:5]
        self._log(f"Top Risk Drivers Identified: {top_drivers}")

        # 3. Filter Modifiable Features
        modifiable_targets = [f for f in top_drivers if self.is_modifiable(f)]
        self._log(f"Modifiable/Targeted features in top 5: {modifiable_targets}")

        # 4. Sequential Optimization Strategy (Hybrid local search)
        X_opt = X.copy()
        current_risk = original_risk
        sequential_mods = {}

        for feat in modifiable_targets:
            orig_val = X.at[X.index[0], feat]
            candidates = self.generate_safe_candidates(feat, orig_val)
            self._log(f"Optimizing {feat} (Current = {orig_val:.2f}) -> Testing cands: {candidates}")
            
            best_cand_risk = current_risk
            best_cand_val = orig_val
            
            for c in candidates:
                X_test = X_opt.copy()
                X_test.at[X_test.index[0], feat] = c
                r = self.predict_risk(X_test)
                if r < best_cand_risk:
                    best_cand_risk = r
                    best_cand_val = c

            if best_cand_risk < current_risk:
                self._log(f"-> Accepted sequential mod: {feat} reduced risk to {best_cand_risk:.4f}")
                X_opt.at[X_opt.index[0], feat] = best_cand_val
                current_risk = best_cand_risk
                sequential_mods[feat] = {"original": float(orig_val), "optimized": float(best_cand_val)}

        # 5. Optional Bounded Combinatorial Exploration (Top 2)
        # We check if altering the top 2 simultaneously performs better than the greedy sequential path.
        final_risk = current_risk
        final_mods = dict(sequential_mods)

        if len(modifiable_targets) >= 2:
            f1, f2 = modifiable_targets[0], modifiable_targets[1]
            val1 = X.at[X.index[0], f1]
            val2 = X.at[X.index[0], f2]
            
            cands1 = self.generate_safe_candidates(f1, val1) or [val1]
            cands2 = self.generate_safe_candidates(f2, val2) or [val2]
            
            best_combo_risk = original_risk # Evaluate against baseline entirely
            combo_mods = {}
            
            for c1 in cands1:
                for c2 in cands2:
                    X_combo = X.copy()
                    if c1 != val1: X_combo.at[X_combo.index[0], f1] = c1
                    if c2 != val2: X_combo.at[X_combo.index[0], f2] = c2
                    
                    c_risk = self.predict_risk(X_combo)
                    if c_risk < best_combo_risk and c_risk < final_risk:
                        best_combo_risk = c_risk
                        combo_mods = {}
                        if c1 != val1: combo_mods[f1] = {"original": float(val1), "optimized": float(c1)}
                        if c2 != val2: combo_mods[f2] = {"original": float(val2), "optimized": float(c2)}
                        
            if best_combo_risk < final_risk:
                self._log("Combinatorial search yielded vastly superior results. Adopting multi-dimensional path.")
                final_risk = best_combo_risk
                final_mods = combo_mods

        # 6. Formatting Outputs
        risk_reduction = original_risk - final_risk
        
        recommended_actions = []
        for f, details in final_mods.items():
            o, opt = details["original"], details["optimized"]
            if o != 0:
                pct_change = int(round(abs(o - opt) / o * 100))
                direction = "Decrease" if opt < o else "Increase"
                recommended_actions.append(f"{direction} {f} by {pct_change}% (from {o:.2f} to {opt:.2f})")

        if recommended_actions:
            clin_summary = f"Pipeline identified {len(final_mods)} clinically safe interventions yielding a {-risk_reduction*100:.1f}% risk deduction. Non-modifiable drivers were bypassed."
        else:
            clin_summary = "No safe interventions available per strict pharmacological and physiological constraints."

        return {
            "original_risk": round(float(original_risk), 4),
            "optimized_risk": round(float(final_risk), 4),
            "risk_reduction": round(float(risk_reduction), 4),
            "recommended_actions": recommended_actions,
            "modified_features": final_mods,
            "top_features": top_drivers,
            "clinical_summary": clin_summary
        }


In [53]:
def generate_clinical_insight(result: Dict[str, Any], api_key: str = None) -> str:
    if not api_key:
        api_key = os.environ.get('GROQ_KEY', None)
    if not api_key:
        return "[Notice]: LLM Translation bypassed. Set GROQ_KEY environment variable for clinical narrative."
        
    # Determine reasoning for low reduction
    low_reduction_reasoning = ""
    if result['risk_reduction'] < 0.05:
        low_reduction_reasoning = "\nNote: Risk reduction is limited as the primary drivers (e.g., age, clinical context) are non-modifiable. Focus on supportive care."

    prompt = f"""You are an expert ICU clinical decision support AI.

Patient ID: {result.get('patient_id', 'N/A')}
Risk Data:
- Original Risk: {result['original_risk']*100:.1f}%
- Optimized Risk: {result['optimized_risk']*100:.1f}%
- Reduction: {result['risk_reduction']*100:.1f}%
- Top Drivers: {result['top_features']}
- Recommended Actions: {result['recommended_actions']}
{low_reduction_reasoning}

Generate a structured clinical alert strictly using these headers:
**RISK SUMMARY**
(Explain the current score and why it is high/low based on the patient's data)

**KEY CONCERNS**
(List 3-4 specific clinical factors from the top drivers contributing to risk)

**RECOMMENDED ACTIONS**
(Provide numbered actionable ICU interventions corresponding to the modifiable features found)

**MONITORING NOTE**
(Bedside monitoring advice)

Be concise, professional, and use medical nomenclature. Do not use generic filler text.
"""
    try:
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": "You are a methodical ICU AI focused strictly on structured patient alerts."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[Notice]: LLM generation failed ({str(e)})"


In [54]:
from sklearn.model_selection import train_test_split

# Load dataset and extract X_test & y_test
def get_test_samples(data_path='deliriumwatch_clean_V5.csv'):
    df = pd.read_csv(data_path)
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    
    X = df.drop(columns=['delirium_label'], errors='ignore')
    y = df['delirium_label'] if 'delirium_label' in df.columns else None
    
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    return X_test, y_test

# Get first 10 patients
X_test_all, y_test_all = get_test_samples()
X_test_subset = X_test_all.head(10)
y_test_subset = y_test_all.head(10)
print(f"Loaded first 10 patients from X_test.")


Loaded first 10 patients from X_test.


In [55]:
def execute_pipeline_multi(test_set, labels):
    logger.info("Instantiating Optimization Agent")
    agent = DeliriumOptimizationAgent(model, debug=False)
    
    all_results = []
    for (idx, row), label in zip(test_set.iterrows(), labels):
        logger.info(f"Processing Patient ID: {idx}")
        patient_df = pd.DataFrame([row])
        result = agent.process_patient(patient_df)
        result['patient_id'] = idx
        result['actual_outcome'] = "DELIRIUM" if label == 1 else "NO DELIRIUM"
        all_results.append(result)
    
    return all_results

results_list = execute_pipeline_multi(X_test_subset, y_test_subset)

2026-03-30 23:33:10 [INFO] Instantiating Optimization Agent
2026-03-30 23:33:10 [INFO] Processing Patient ID: 23762
2026-03-30 23:33:10 [INFO] Processing Patient ID: 6172
2026-03-30 23:33:10 [INFO] Processing Patient ID: 10145
2026-03-30 23:33:10 [INFO] Processing Patient ID: 10011
2026-03-30 23:33:11 [INFO] Processing Patient ID: 6981
2026-03-30 23:33:11 [INFO] Processing Patient ID: 6420
2026-03-30 23:33:11 [INFO] Processing Patient ID: 6506
2026-03-30 23:33:11 [INFO] Processing Patient ID: 24760
2026-03-30 23:33:11 [INFO] Processing Patient ID: 22166
2026-03-30 23:33:11 [INFO] Processing Patient ID: 5005


In [56]:
import json
from IPython.display import display, Markdown

def get_risk_label(risk):
    if risk < 0.3: return "🟢 LOW"
    if risk < 0.6: return "🟡 MEDIUM"
    return "🔴 HIGH"

for r in results_list:
    # 1. Print Raw JSON
    print(f"\n{'='*50}\nRAW RESULT - PATIENT {r['patient_id']}\n{'='*50}")
    print(json.dumps(r, indent=2))
    
    # 2. Print Formatted Clinical Alert
    print(f"\n{'='*50}")
    print(f"CLINICAL ALERT - PATIENT {r['patient_id']}")
    print(f"{'='*50}")
    
    risk_level = get_risk_label(r['original_risk'])
    outcome_emoji = "✅" if r['actual_outcome'] == "NO DELIRIUM" else "⚠️"
    
    print(f"Risk Level: {risk_level} ({r['original_risk']*100:.1f}%)")
    print(f"Actual Outcome: {outcome_emoji} {r['actual_outcome']}")
    
    # Generate and print LLM narrative
    explanation = generate_clinical_insight(r)
    print(f"\n{explanation}")
    print(f"\n{'='*50}\n")


RAW RESULT - PATIENT 23762
{
  "original_risk": 0.9693,
  "optimized_risk": 0.9568,
  "risk_reduction": 0.0124,
  "recommended_actions": [
    "Decrease midazolam_total by 30% (from 397.42 to 278.19)",
    "Decrease fentanyl_total by 30% (from 436.44 to 305.51)"
  ],
  "modified_features": {
    "midazolam_total": {
      "original": 397.42,
      "optimized": 278.194
    },
    "fentanyl_total": {
      "original": 436.44,
      "optimized": 305.508
    }
  },
  "top_features": [
    "propofol_total",
    "midazolam_total",
    "fentanyl_total",
    "hr_std",
    "unique_sedation_drugs"
  ],
  "clinical_summary": "Pipeline identified 2 clinically safe interventions yielding a -1.2% risk deduction. Non-modifiable drivers were bypassed.",
  "patient_id": 23762,
  "actual_outcome": "DELIRIUM"
}

CLINICAL ALERT - PATIENT 23762
Risk Level: 🔴 HIGH (96.9%)
Actual Outcome: ⚠️ DELIRIUM


2026-03-30 23:33:12 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



**RISK SUMMARY**
Patient 23762's optimized risk score is 95.7%, indicating a high risk of adverse outcomes. The 1.2% reduction in risk is limited due to non-modifiable primary drivers. The current risk score is high due to the patient's high sedation burden and hemodynamic instability.

**KEY CONCERNS**
1. **Propofol_total**: Elevated propofol dosing contributing to sedation burden.
2. **Midazolam_total**: High midazolam dosing contributing to sedation burden.
3. **Fentanyl_total**: Elevated fentanyl dosing contributing to sedation burden and potential hemodynamic instability.
4. **HR_std**: Elevated heart rate variability indicating potential hemodynamic instability.

**RECOMMENDED ACTIONS**
1. Decrease midazolam_total by 30% (from 397.42 to 278.19) to reduce sedation burden.
2. Decrease fentanyl_total by 30% (from 436.44 to 305.51) to reduce sedation burden and potential hemodynamic instability.

**MONITORING NOTE**
Closely monitor the patient's hemodynamic status, including heart r

2026-03-30 23:33:14 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



**RISK SUMMARY**
Patient 6172's ICU risk score remains elevated at 57.3% (Original and Optimized). The lack of risk reduction is attributed to non-modifiable primary drivers, including advanced age and clinical context.

**KEY CONCERNS**
1. **Age**:  Patient's advanced age (likely > 65 years) contributes significantly to the elevated risk score.
2. **Avg SOFA Score**: The patient's average Sequential Organ Failure Assessment (SOFA) score indicates multi-organ dysfunction, increasing the risk of mortality.
3. **SpO2 Min**: The patient's lowest oxygen saturation (SpO2) level is below 90%, indicating potential hypoxemia.
4. **RR Std**: The patient's respiratory rate standard deviation (RR Std) suggests tachypnea, which may be a sign of respiratory distress.

**RECOMMENDED ACTIONS**
1. **Optimize Oxygen Therapy**: Titrate oxygen to maintain SpO2 > 92% to minimize hypoxemia.
2. **Monitor and Manage Tachypnea**: Closely monitor RR Std and consider non-invasive ventilation (NIV) or invasive 

2026-03-30 23:33:16 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


... (11 outputs left out)

2026-03-30 23:33:23 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



**RISK SUMMARY**
Patient 22166's optimized risk score has decreased from 90.8% to 34.6% (reduction of 56.2%), indicating a significant improvement in clinical stability. This reduction is attributed to the optimization of sedation and the patient's hemodynamic status.

**KEY CONCERNS**
1. **Propofol Total**: Elevated propofol dosing (13603.57 mg) is a significant contributor to the patient's risk.
2. **NID Score**: A high NID score (0.30) suggests a high risk of nosocomial infections.
3. **Age**: The patient's advanced age is a contributing factor to their risk.
4. **HR Std**: Elevated heart rate variability (HR Std) indicates potential hemodynamic instability.

**RECOMMENDED ACTIONS**
1. Decrease propofol_total by 30% (from 13603.57 to 9522.50 mg) to optimize sedation and reduce risk.
2. Decrease NID score by 20% (from 0.30 to 0.24) through implementation of infection control measures.

**MONITORING NOTE**
Closely monitor the patient's hemodynamic status, including heart rate and blo

2026-03-30 23:33:25 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



**RISK SUMMARY**
Patient 5005's optimized risk score has decreased from 61.1% to 43.3%, indicating a reduction of 17.8%. This improvement is attributed to the optimization of various vital signs and hemodynamic parameters.

**KEY CONCERNS**
1. Elevated NID score: Contributing to increased risk
2. Hypertension (nibp_mean 99.93 mmHg): Exceeding normal limits
3. Hypoxemia (spo2_mean 91.90%): Below target range
4. Tachycardia (hr_mean 92.33 bpm): Above normal limits

**RECOMMENDED ACTIONS**
1. Gradually decrease mean arterial blood pressure (nibp_mean) by 5% to 94.93 mmHg
2. Increase mean oxygen saturation (spo2_mean) by 5% to 96.50%
3. Decrease mean heart rate (hr_mean) by 15% to 78.48 bpm

**MONITORING NOTE**
Closely monitor the patient's vital signs, including blood pressure, oxygen saturation, and heart rate, to ensure the recommended interventions are effective and do not cause adverse effects.


